In [1]:
%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
import glob
from reprojection import *
from show import *
from utils import *
import pandas as pd
from scipy.signal import convolve2d

In [2]:
files = sorted(glob.glob('/home/ulyanov/data/solo/phi/nonpolar/*'))
print(len(files))
print(files[0], files[-1])

220
/home/ulyanov/data/solo/phi/nonpolar/solo_L2_phi-fdt-blos_20241101T001503_V202602220112_0451010501.fits.gz /home/ulyanov/data/solo/phi/nonpolar/solo_L2_phi-fdt-blos_20241130T211503_V202602220151_0451300508.fits.gz


In [3]:
df = pd.read_csv('/home/ulyanov/data/solo/phi/wcs/fdt/disk_centers.csv', skipinitialspace=True).dropna()

xu_sun, yu_sun, ru_sun = df['xu_sun'].to_numpy(), df['yu_sun'].to_numpy(), df['ru_sun'].to_numpy()
dids = df['did'].to_numpy()

In [4]:
s = np.load('/home/ulyanov/data/solo/phi/cross_calibration/fdt/hmi.npz')
fdt, hmi, psf = s['fdt'], s['hmi'], s['psf']

s = np.load('/home/ulyanov/data/solo/phi/distortion/fdt/distortion.npz')
xd, yd = s['xd'], s['yd']

In [7]:
nx, ny = 1024, 1024
xc, yc = 511.5, 511.5
Rsun = 512

view_north = View(nx, ny, xc, yc, Rsun, -90, 90, 0) ### North pole view
grid = np.mgrid[:nx,:ny].astype(np.float32)

mean, variance, coverage = np.zeros_like(grid[0]), np.zeros_like(grid[0]), np.zeros_like(grid[0])

plt.ioff()

for file in files[:]:
    print(file)

    with fits.open(file) as hdul:
        header = hdul[0].header.copy()
        data = hdul[0].data.copy()

    did = int(file.split('_')[-1].split('.')[0])
    data = undistort(data, header, xd, yd)
    #data = hmize(data, psf)#, fdt, hmi)

    view = View.from_header(header)
    view.update(xc=xu_sun[dids == did][0], yc=yu_sun[dids == did][0], crota=view.crota + 0.3, rsun=ru_sun[dids == did][0] + 0.12)

    transform = view_north.to_spherical(correct_mu=True, mu_thr=0.1) - view.to_spherical(correct_mu=True, correct_dr=True, mu_thr=0.1, stonyhurst=False)
    grid_, alpha = transform(grid)
    data = bilinear(data, *grid_) * alpha

    n = (~np.isnan(data)) * (1 / alpha) ** 4
    coverage += np.nan_to_num(n)
    mean_ = mean.copy()
    mean += np.nan_to_num((data - mean) * n / coverage)
    variance += np.nan_to_num((data - mean) * (data - mean_) * n)

    #show_polar_data(data, vmin=-50, vmax=50, cmap='seismic', label=r'$B_{los}, G$')
    #plt.savefig(f'temp/{file.split("/")[-1].split(".")[0]}.png')
    #plt.savefig(f'temp/{file.split("/")[-1].split(".")[2]}.png')
    #plt.close()

plt.ion()

variance = variance / coverage
mean[coverage == 0] = np.nan
coverage[coverage == 0] = np.nan
variance[coverage == 0] = np.nan

/home/ulyanov/data/solo/phi/nonpolar/solo_L2_phi-fdt-blos_20241101T001503_V202602220112_0451010501.fits.gz
/home/ulyanov/data/solo/phi/nonpolar/solo_L2_phi-fdt-blos_20241101T031503_V202602220112_0451010502.fits.gz
/home/ulyanov/data/solo/phi/nonpolar/solo_L2_phi-fdt-blos_20241101T061503_V202602220112_0451010503.fits.gz
/home/ulyanov/data/solo/phi/nonpolar/solo_L2_phi-fdt-blos_20241101T091502_V202602220112_0451010504.fits.gz
/home/ulyanov/data/solo/phi/nonpolar/solo_L2_phi-fdt-blos_20241101T121503_V202602220112_0451010505.fits.gz
/home/ulyanov/data/solo/phi/nonpolar/solo_L2_phi-fdt-blos_20241101T151503_V202602220112_0451010506.fits.gz
/home/ulyanov/data/solo/phi/nonpolar/solo_L2_phi-fdt-blos_20241101T181503_V202602220112_0451010507.fits.gz
/home/ulyanov/data/solo/phi/nonpolar/solo_L2_phi-fdt-blos_20241101T211503_V202602220112_0451010508.fits.gz
/home/ulyanov/data/solo/phi/nonpolar/solo_L2_phi-fdt-blos_20241102T001503_V202602220112_0451020501.fits.gz
/home/ulyanov/data/solo/phi/nonpolar/

/tmp/ipykernel_34945/3642014821.py:43: RuntimeWarning: invalid value encountered in divide
  variance = variance / coverage


In [8]:
show_polar_data(mean * 1.3, vmin=-25, vmax=25, cmap='seismic', label=r'$B_{los}, G$')

(<Figure size 1000x1000 with 2 Axes>, <Axes: >)

In [7]:
show_polar_data(np.sqrt(variance), vmin=0, vmax=20, cmap='turbo', label=r'$\sigma_B, G$')

(<Figure size 1000x1000 with 2 Axes>, <Axes: >)

In [5]:
fdt = mean.copy()

In [7]:
plt.figure(figsize=(10,10))
plt.imshow(alpha)

In [31]:
plt.figure(figsize=(10,10))
plt.scatter(mean, fdt, s=0.1)
plt.xlim(-50,50)
plt.ylim(-50,50)

(-50.0, 50.0)

In [34]:
dx = 0.5

x = []
q = []
for x_ in np.arange(-10,10,dx):
    a = x_ ** 3
    b = (x_ + dx) ** 3

    t = np.where(np.all([mean > a, mean < b], axis=0))
    x += [(a + b) / 2]
    q += [np.mean(fdt[t])]

q = np.array(q)

plt.figure(figsize=(10,10))
plt.scatter(mean, fdt, s=0.1)
plt.plot(x, q, color='black')

plt.xlim(-200,200)
plt.ylim(-200,200)

plt.grid(True)
plt.tight_layout()

In [5]:
fdt = mean.copy()

In [6]:
9 / 360 * 27.27 * 24

16.362000000000002

In [9]:
data = mean
data_ = fdt * 1.2
xi, yi = (grid[0] - xc) / Rsun, (grid[1] - yc) / Rsun
r2 = xi ** 2 + yi ** 2

delta_x, delta_y = np.gradient((data + data_) / 2)

delta = delta_x * yi - delta_y * xi
delta_t = data - data_

W = delta_t / delta / Rsun * 180 / np.pi

In [10]:
plt.figure(figsize=(10,10))
plt.imshow(W, vmin=-10, vmax=10)

In [11]:
dr = 0.05
ri = np.arange(0, 1, dr)
Q = []

for r in ri:
    t = np.where(np.abs(r2 - (r + dr / 2)) < dr / 2)

    a = delta[t] ** 2
    b = delta_t[t] * delta[t]

    q = np.mean(b) / np.mean(a)
    Q += [q]

Q = np.array(Q) / Rsun * 180 / np.pi

plt.figure(figsize=(10,8))
plt.plot(1 - (ri + dr / 2), Q / 0.68)

In [7]:
10 * 14 * 24 * 3600

12096000

In [9]:
np.tan(header['RSUN_ARC'] / 3600 * np.pi / 180)

np.float64(0.005276201104729802)